In [6]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# -------------------------------------
# ACCOUNTS FOR AUTO SWITCHING
# -------------------------------------
ACCOUNTS = [
    ("man411210@gmail.com", "Click@123"),
    ("man41121.0@gmail.com", "Click@123"),
    ("man4112.10@gmail.com", "Click@123"),
    ("man4112.1.0@gmail.com", "Click@123"),
    ("man411.210@gmail.com", "Click@123"),
]

current_account_index = 0

# -------------------------------------
# FOLDERS
# -------------------------------------
os.makedirs("done", exist_ok=True)
os.makedirs("html_dumps", exist_ok=True)
os.makedirs("csv", exist_ok=True)

# -------------------------------------
# LOGIN
# -------------------------------------
def login_totalcarcheck(driver, email, password):
    driver.get("https://totalcarcheck.co.uk/Account/Login")
    WebDriverWait(driver, 12).until(
        EC.presence_of_element_located((By.ID, "UserName"))
    )

    driver.find_element(By.ID, "UserName").clear()
    driver.find_element(By.ID, "Password").clear()
    driver.find_element(By.ID, "UserName").send_keys(email)
    driver.find_element(By.ID, "Password").send_keys(password)
    driver.find_element(By.CSS_SELECTOR, "input.btn.btn-primary").click()

    WebDriverWait(driver, 12).until(
        EC.presence_of_element_located((By.ID, "userIdLink"))
    )

    print(f"✅ Logged in: {email}")


# -------------------------------------
# PROPER LOGOUT FIXED
# -------------------------------------
def logout_properly(driver):
    print("🔐 Logging out...")

    try:
        driver.get("https://totalcarcheck.co.uk/Account/DoYouWantToLogOut")
        time.sleep(1)

        logout_button = WebDriverWait(driver, 12).until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, "button.btn.btn-danger.btn-lg")
            )
        )
        logout_button.click()

        print("✅ Successfully logged out")
        time.sleep(2)

    except Exception as e:
        print(f"❌ Logout failed: {e}")


# -------------------------------------
# SWITCH ACCOUNT
# -------------------------------------
def switch_account(driver):
    global current_account_index

    current_account_index += 1
    if current_account_index >= len(ACCOUNTS):
        current_account_index = 0

    email, pw = ACCOUNTS[current_account_index]

    print(f"🔄 Switching account → {email}")

    logout_properly(driver)
    login_totalcarcheck(driver, email, pw)


# -------------------------------------
# START DRIVER
# -------------------------------------
def start_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_experimental_option("detach", True)

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    email, pw = ACCOUNTS[current_account_index]
    login_totalcarcheck(driver, email, pw)

    return driver


driver = start_driver()


# -------------------------------------
# SAVE HTML (CSV SUBFOLDER SUPPORT)
# -------------------------------------
def save_html(reg, html, folder):
    path = f"{folder}/{reg.replace(' ', '_')}.html"
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"💾 HTML saved: {path}")


# -------------------------------------
# SCRAPE SINGLE REG
# -------------------------------------
def fetch_vehicle_info(reg, driver, html_folder_path):
    url = f"https://totalcarcheck.co.uk/FreeCheck?regno={reg.replace(' ', '+')}"

    while True:
        driver.get(url)
        time.sleep(1)

        # --- RATE LIMIT CHECK ---
        try:
            rate = driver.find_element(
                By.XPATH,
                "//pre[contains(text(),'too many vehicles')]"
            )
            if rate.is_displayed():
                print("⚠️ Rate-limit → Switching account...")
                save_html(reg, driver.page_source, html_folder_path)
                switch_account(driver)
                continue
        except:
            pass

        # --- NORMAL DATA EXTRACT ---
        try:
            mot = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located(
                    (By.XPATH, '//span[text()="MOT Status"]/following::span[1]')
                )
            ).text.strip()

            tax = driver.find_element(
                By.XPATH,
                '//span[text()="Road Tax Status"]/following::span[1]'
            ).text.strip()

            save_html(reg, driver.page_source, html_folder_path)

            return {"Reg": reg, "MOT": mot, "TAX": tax}

        except Exception as e:
            print(f"❌ Failed to extract for {reg}: {e}")
            save_html(reg, driver.page_source, html_folder_path)
            return {"Reg": reg, "MOT": "", "TAX": ""}


# -------------------------------------
# PROCESS ALL CSVs INSIDE /csv
# -------------------------------------
all_results = []

for file in os.listdir("csv"):
    if file.endswith(".csv"):
        path = os.path.join("csv", file)
        print(f"\n📂 Reading → {path}")

        df = pd.read_csv(path)

        if "Reg" not in df.columns:
            print("⚠️ No 'Reg' column found, skipping...")
            continue

        # Create HTML folder for this CSV
        csv_folder_name = os.path.splitext(file)[0]
        html_folder_path = f"html_dumps/{csv_folder_name}"
        os.makedirs(html_folder_path, exist_ok=True)

        for reg in df["Reg"].dropna().unique():
            print(f"🔍 Fetching info: {reg}")
            info = fetch_vehicle_info(reg, driver, html_folder_path)
            all_results.append(info)
            time.sleep(1)

# -------------------------------------
# SAVE FINAL OUTPUT
# -------------------------------------
output_df = pd.DataFrame(all_results)
output_df.to_csv("done/output.csv", index=False)

print("\n🎉 All done! Saved → done/output.csv")


✅ Logged in: man411210@gmail.com

📂 Reading → csv\Final_Barclay_cleaned.csv
🔍 Fetching info: K100RMF
⚠️ Rate-limit → Switching account...
💾 HTML saved: html_dumps/Final_Barclay_cleaned/K100RMF.html
🔄 Switching account → man41121.0@gmail.com
🔐 Logging out...
✅ Successfully logged out
✅ Logged in: man41121.0@gmail.com
💾 HTML saved: html_dumps/Final_Barclay_cleaned/K100RMF.html
🔍 Fetching info: CE61KWA
💾 HTML saved: html_dumps/Final_Barclay_cleaned/CE61KWA.html
🔍 Fetching info: WU06RFO
💾 HTML saved: html_dumps/Final_Barclay_cleaned/WU06RFO.html
🔍 Fetching info: KU63EZN
💾 HTML saved: html_dumps/Final_Barclay_cleaned/KU63EZN.html
🔍 Fetching info: PN15BKF
💾 HTML saved: html_dumps/Final_Barclay_cleaned/PN15BKF.html
🔍 Fetching info: HG56BHZ
💾 HTML saved: html_dumps/Final_Barclay_cleaned/HG56BHZ.html
🔍 Fetching info: VA11EWH
💾 HTML saved: html_dumps/Final_Barclay_cleaned/VA11EWH.html
🔍 Fetching info: LA69KUK
💾 HTML saved: html_dumps/Final_Barclay_cleaned/LA69KUK.html
🔍 Fetching info: EX16ZBJ
💾 